Extract data from csv

In [238]:
import pandas as pd
import numpy as np

In [239]:
df = pd.read_csv('data/bronze/dirty_cafe_sales.csv')

In [240]:
df.sample(5)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1570,TXN_9969283,Sandwich,3,4.0,12.0,Digital Wallet,NaN,2023-10-26
996,TXN_9868382,Coffee,2,2.0,4.0,Digital Wallet,NaN,2023-06-02
2173,TXN_8397155,Salad,4,5.0,20.0,Credit Card,Takeaway,2023-08-16
1060,TXN_6732989,Salad,2,5.0,10.0,Credit Card,NaN,2023-04-29
7928,TXN_7308629,Salad,4,5.0,20.0,ERROR,Takeaway,2023-11-24


In [241]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [242]:
df_copy = df.copy()

In [243]:
df_copy.columns = [c.lower().replace(' ', '_') for c in df_copy.columns]

In [244]:
df_copy.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [245]:
df_copy['item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

In [246]:
df_copy['is_error'] = False

In [247]:
df_copy['is_missing'] = False

In [ ]:
def is_missing_flag(df, columns, value): 
    for column in columns:
        df.loc[(df[column] == value) | (df[column].isna()), 'is_missing'] = True

In [ ]:
def is_error_flag(df, columns, value): 
    for column in columns:
        df.loc[df_copy[column] == value, 'is_error'] = True

In [250]:
columns_for_missing_flag = ['item', 'quantity', 'price_per_unit', 'total_spent', 'transaction_date']
is_missing_flag(df_copy, columns_for_missing_flag, 'UNKNOWN')

In [251]:
columns_for_error_flag = ['item', 'quantity', 'price_per_unit', 'total_spent']
is_error_flag(df_copy, columns_for_error_flag, 'ERROR')

In [252]:
df_copy[df_copy['is_missing'] == True]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02,False,True
31,TXN_8927252,UNKNOWN,2,1.0,ERROR,Credit Card,ERROR,2023-11-06,True,True
33,TXN_7710508,UNKNOWN,5,1.0,5.0,Cash,NaN,ERROR,False,True
...,...,...,...,...,...,...,...,...,...,...
9961,TXN_2153100,Tea,2,UNKNOWN,3.0,Cash,NaN,2023-12-29,False,True
9983,TXN_9226047,Smoothie,3,4.0,12.0,Cash,NaN,UNKNOWN,False,True
9984,TXN_3142496,Smoothie,UNKNOWN,4.0,4.0,Cash,Takeaway,2023-07-27,False,True
9994,TXN_7851634,UNKNOWN,4,4.0,16.0,NaN,NaN,2023-01-08,False,True


In [277]:
df_copy.replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

In [254]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    10000 non-null  object
 1   item              9031 non-null   object
 2   quantity          9521 non-null   object
 3   price_per_unit    9467 non-null   object
 4   total_spent       9498 non-null   object
 5   payment_method    6822 non-null   object
 6   location          6039 non-null   object
 7   transaction_date  9540 non-null   object
 8   is_error          10000 non-null  bool  
 9   is_missing        10000 non-null  bool  
dtypes: bool(2), object(8)
memory usage: 644.7+ KB


In [255]:
df_copy['price_per_unit'] = df_copy['price_per_unit'].astype(float)

In [256]:
df_copy['total_spent'] = df_copy['total_spent'].astype(float)

In [257]:
df_copy['quantity'] = pd.to_numeric(df_copy['quantity'], errors='coerce').astype('Int64')

In [258]:
df_copy['transaction_date'] = pd.to_datetime(df_copy['transaction_date'])

In [259]:
type(df_copy['quantity'][0])

numpy.int64

In [260]:
df_copy.loc[
   (df_copy['price_per_unit'].isna() &
    df_copy['total_spent'].notna() &
    df_copy['quantity'].notna()
    ), 'price_per_unit'] = df_copy['total_spent']/df_copy['quantity']

In [262]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9031 non-null   object        
 2   quantity          9521 non-null   Int64         
 3   price_per_unit    9962 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   payment_method    6822 non-null   object        
 6   location          6039 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
 8   is_error          10000 non-null  bool          
 9   is_missing        10000 non-null  bool          
dtypes: Int64(1), bool(2), datetime64[ns](1), float64(2), object(4)
memory usage: 654.4+ KB


In [263]:
df_copy['item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', nan, 'Sandwich',
       'Juice', 'Tea'], dtype=object)

In [264]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
14,TXN_8915701,NaN,2,1.5,3.0,NaN,In-store,2023-03-21,True,False
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02,False,True
31,TXN_8927252,NaN,2,1.0,NaN,Credit Card,NaN,2023-11-06,True,True
...,...,...,...,...,...,...,...,...,...,...
9951,TXN_4122925,NaN,4,1.0,4.0,NaN,Takeaway,2023-10-20,True,False
9958,TXN_4125474,NaN,2,5.0,10.0,Credit Card,In-store,2023-08-02,True,False
9981,TXN_4583012,NaN,5,4.0,20.0,Digital Wallet,NaN,2023-02-27,True,False
9994,TXN_7851634,NaN,4,4.0,16.0,NaN,NaN,2023-01-08,False,True


In [265]:
mapping_df = df_copy.loc[
   (df_copy['item'].notna()) & 
   (df_copy['price_per_unit'].notna()),
   ['item', 'price_per_unit']].drop_duplicates()

In [266]:
mapping_price_dict = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [267]:
mapping_price_dict

{'Coffee': 2.0,
 'Cake': 3.0,
 'Cookie': 1.0,
 'Salad': 5.0,
 'Smoothie': 4.0,
 'Sandwich': 4.0,
 'Juice': 3.0,
 'Tea': 1.5}

In [268]:
mapping_df.drop_duplicates(subset=['price_per_unit'], keep=False, inplace=True)

In [269]:
mapping_item_dict = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [270]:
mapping_item_dict

{'Coffee': 2.0, 'Cookie': 1.0, 'Salad': 5.0, 'Tea': 1.5}

In [271]:
for item, price in mapping_item_dict.items():
   mask = (df_copy['item'].isna()) & (df_copy['price_per_unit'] == price)
   df_copy.loc[mask, 'item'] = item

In [272]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
36,TXN_6855453,NaN,4,3.0,12.0,NaN,In-store,2023-07-17,False,True
61,TXN_8051289,NaN,1,3.0,3.0,NaN,In-store,2023-10-09,False,True
69,TXN_8471743,NaN,5,3.0,15.0,Digital Wallet,In-store,2023-04-06,True,False
...,...,...,...,...,...,...,...,...,...,...
9910,TXN_2338617,NaN,2,3.0,6.0,Digital Wallet,NaN,2023-01-12,True,False
9918,TXN_2292088,NaN,1,4.0,4.0,Digital Wallet,Takeaway,2023-03-04,True,False
9946,TXN_8807600,NaN,1,4.0,4.0,Cash,Takeaway,2023-09-24,False,True
9981,TXN_4583012,NaN,5,4.0,20.0,Digital Wallet,NaN,2023-02-27,True,False


In [273]:
for item, price in mapping_price_dict.items():
   mask = (df_copy['price_per_unit'].isna()) & (df_copy['item'] == item)
   df_copy.loc[mask, 'price_per_unit'] = price

In [274]:
df_copy.loc[
   (df_copy['price_per_unit'].notna()) &
   (df_copy['total_spent'].isna()) &
   (df_copy['quantity'].notna()
   ), 'total_spent'] = df_copy['price_per_unit'] * df_copy['quantity']

In [275]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9520 non-null   object        
 2   quantity          9521 non-null   Int64         
 3   price_per_unit    9994 non-null   float64       
 4   total_spent       9977 non-null   float64       
 5   payment_method    6822 non-null   object        
 6   location          6039 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
 8   is_error          10000 non-null  bool          
 9   is_missing        10000 non-null  bool          
dtypes: Int64(1), bool(2), datetime64[ns](1), float64(2), object(4)
memory usage: 654.4+ KB


In [276]:
df_copy[df_copy['total_spent'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
236,TXN_8562645,Salad,<NA>,5.0,NaN,NaN,In-store,2023-05-18,True,True
278,TXN_3229409,Juice,<NA>,3.0,NaN,Cash,Takeaway,2023-04-15,False,True
641,TXN_2962976,Juice,<NA>,3.0,NaN,NaN,NaN,2023-03-17,True,True
738,TXN_8696094,Sandwich,<NA>,4.0,NaN,NaN,Takeaway,2023-05-14,False,True
1761,TXN_3611851,NaN,4,NaN,NaN,Credit Card,NaN,2023-02-09,True,True
2289,TXN_7524977,NaN,4,NaN,NaN,NaN,NaN,2023-12-09,False,True
2796,TXN_9188692,Cake,<NA>,3.0,NaN,Credit Card,NaN,2023-12-01,True,True
3203,TXN_4565754,Smoothie,<NA>,4.0,NaN,Digital Wallet,Takeaway,2023-10-06,True,False
3224,TXN_6297232,Coffee,<NA>,2.0,NaN,NaN,NaN,2023-04-07,False,True
3401,TXN_3251829,Tea,<NA>,1.5,NaN,Digital Wallet,In-store,2023-07-25,False,False
